# 会话摘要记忆 · Running Summary

**本 Notebook 目标：** 演示如何用 **LLM 压缩长对话**为运行摘要（running summary），在 Token 预算内保留尽可能多的任务有效信息。

> 每轮对话后直接调用 **OpenAI 兼容 API**（DeepSeek / OpenAI）更新摘要，用摘要替代完整历史注入 Prompt。不依赖 LangChain。

## 概念：为什么要摘要？

滑动窗口（`01-windowed_memory`）**丢弃**旧消息；摘要记忆**压缩**旧消息——保留语义，减少 Token。

| 策略 | 旧信息处理 | 优点 | 缺点 |
|------|-----------|------|------|
| 滑动窗口 | 直接丢弃 | 简单、无额外 LLM 成本 | 早期关键信息可能丢失 |
| **摘要记忆** | LLM 压缩为摘要 | 保留远程上下文语义 | 每轮多一次 LLM 调用；摘要可能丢细节 |
| 长期记忆（`02_long_term_memory`） | 向量库持久化 + 检索 | 跨会话、可语义召回 | 需维护向量库与检索逻辑 |

## 整体架构

```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
flowchart TB
    subgraph Memory["SummaryMemory"]
        BUF["buffer: 当前累积摘要"]
        API["OpenAI SDK\nDeepSeek / OpenAI"]
    end

    SC["save_context(input, output)"] --> PROMPT["构造中文摘要 Prompt"]
    PROMPT --> API
    API --> BUF
    BUF --> LMV["load_memory_variables()"]
    LMV --> INJECT["注入 system prompt\n{history: 摘要...}"]
```

## 三种记忆策略对比

```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
flowchart LR
    W["01 滑动窗口<br/>保留最近 N 条 → pop 最早消息"]
    S["03 摘要记忆<br/>LLM 压缩历史 → buffer 摘要"]
    L["02 长期记忆<br/>向量库存储 → 语义检索召回"]
    W --- S --- L
```

## 生产策略（超出本 Demo 范围）

| 策略 | 适用场景 |
|------|---------|
| **Running Summary**（本课） | 对话轮次中等，每轮增量更新 |
| **Map-Reduce Summary** | 历史极长，先分块摘要再合并 |
| **Refine Summary** | 逐轮 refine 旧摘要，保留更多细节 |
| **Summary + 长期记忆** | 摘要管近期，向量库管跨会话事实 |

> 生产环境可直接复用本课的 `SummaryMemory` 类，或接入 LangGraph checkpoint / Store API。

---
# Part 1 · 环境准备

自动安装 `openai` / `tiktoken` / `python-dotenv`，并加载项目根目录 `.env`。

# Part 2 · 创建 API Client

需配置 `DEEPSEEK_API_KEY` 或 `OPENAI_API_KEY`（项目根目录 `.env`）。使用 OpenAI SDK 直接调用 Chat Completions API。

In [1]:
import os
from openai import OpenAI


def create_client() -> tuple[OpenAI, str]:
    """优先 DeepSeek，其次 OpenAI。返回 (client, model)。"""
    deepseek_key = os.environ.get("DEEPSEEK_API_KEY")
    if deepseek_key:
        print("[LLM] DeepSeek deepseek-chat")
        return OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com"), "deepseek-chat"

    openai_key = os.environ.get("OPENAI_API_KEY")
    if openai_key:
        model = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
        print(f"[LLM] OpenAI {model}")
        return OpenAI(api_key=openai_key), model

    raise RuntimeError("请在 .env 中配置 DEEPSEEK_API_KEY 或 OPENAI_API_KEY")


client, model = create_client()
print(f"[Part 2] API Client 已就绪 | model={model}")

[LLM] DeepSeek deepseek-chat
[Part 2] API Client 已就绪 | model=deepseek-chat


# Part 3 · 摘要记忆类

自己实现 `SummaryMemory`：每轮 `save_context` 构造中文 Prompt，**直接调用 API** 增量更新 `buffer`。

In [2]:
SUMMARY_PROMPT_ZH = """请用中文增量更新对话摘要：在已有摘要基础上，合并本轮新对话，输出一段新的中文摘要。

示例
当前摘要：
用户询问 AI 对人工智能的看法。AI 认为人工智能是积极的力量。

新对话：
用户：你为什么认为人工智能是积极的力量？
助手：因为它能帮助人类发挥潜能。

新摘要：
用户询问 AI 对人工智能的看法。AI 认为人工智能是积极的力量，因为它能帮助人类发挥潜能。
示例结束

要求：摘要必须使用中文，保留关键事实，语言简洁。只输出摘要正文，不要解释。

当前摘要：
{summary}

新对话：
{new_lines}

新摘要："""


class SummaryMemory:
    """Running Summary：每轮对话后调用 LLM API 更新 buffer。"""

    def __init__(self, client: OpenAI, model: str):
        self.client = client
        self.model = model
        self.buffer = ""

    def _call_llm(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return (response.choices[0].message.content or "").strip()

    def save_context(self, inputs: dict, outputs: dict) -> None:
        user_text = inputs["input"]
        assistant_text = outputs["output"]
        new_lines = f"用户：{user_text}\n助手：{assistant_text}"
        prompt = SUMMARY_PROMPT_ZH.format(
            summary=self.buffer or "（暂无）",
            new_lines=new_lines,
        )
        self.buffer = self._call_llm(prompt)

    def load_memory_variables(self, _: dict | None = None) -> dict:
        return {"history": self.buffer}


memory = SummaryMemory(client, model)
_NOTEBOOK_READY = True
print("[Part 3] SummaryMemory 已初始化")


[Part 3] SummaryMemory 已初始化


---
# Part 4 · 多轮对话演示

模拟 3 轮电商客服项目对话。**每轮 `save_context` 直接调 API 更新摘要**，观察 buffer 变化。

In [3]:
if not globals().get("_NOTEBOOK_READY", False):
    raise RuntimeError("请先 Run All，或依次运行 Part 1 → Part 2 → Part 3")

TURNS = [
    (
        {"input": "我在做电商客服项目。"},
        {"output": "好的，需要我帮你梳理架构吗？"},
    ),
    (
        {"input": "我们使用向量库存 FAQ。"},
        {"output": "明白，可再加一层重排。"},
    ),
    (
        {"input": "召回后还要做 MMR 去重。"},
        {"output": "可以，避免重复段落占满上下文。"},
    ),
]

RAW_HISTORY = []
for i, (user_msg, assistant_msg) in enumerate(TURNS, 1):
    memory.save_context(user_msg, assistant_msg)
    RAW_HISTORY.append(f"用户: {user_msg['input']}")
    RAW_HISTORY.append(f"助手: {assistant_msg['output']}")
    print(f"\n{'=' * 60}")
    print(f"第 {i} 轮对话后 · memory.buffer（{len(memory.buffer)} 字）")
    print(f"{'=' * 60}")
    print(memory.buffer)

vars_ = memory.load_memory_variables({})
print(f"\n{'=' * 60}")
print("load_memory_variables({})")
print(f"{'=' * 60}")
print(vars_)

assert "history" in vars_
assert isinstance(vars_["history"], str) and len(vars_["history"]) > 0
assert vars_["history"] == memory.buffer
print("\n[Part 4] 多轮摘要演示通过 ✓")


第 1 轮对话后 · memory.buffer（26 字）
用户在做电商客服项目，助手询问是否需要帮忙梳理架构。

第 2 轮对话后 · memory.buffer（53 字）
用户在做电商客服项目，助手询问是否需要帮忙梳理架构。用户表示使用向量库存 FAQ，助手建议可再加一层重排。

第 3 轮对话后 · memory.buffer（92 字）
用户在做电商客服项目，助手询问是否需要帮忙梳理架构。用户表示使用向量库存 FAQ，助手建议可再加一层重排。用户补充召回后需做 MMR 去重，助手表示同意，认为可避免重复段落占满上下文。

load_memory_variables({})
{'history': '用户在做电商客服项目，助手询问是否需要帮忙梳理架构。用户表示使用向量库存 FAQ，助手建议可再加一层重排。用户补充召回后需做 MMR 去重，助手表示同意，认为可避免重复段落占满上下文。'}

[Part 4] 多轮摘要演示通过 ✓


# Part 5 · Prompt 注入、长度对比与真实推理

对比完整历史 vs 摘要 Token 数，并**直接调 API** 基于摘要回答后续问题：

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

full_history = "\n".join(RAW_HISTORY)
summary = memory.buffer
next_question = "如何做检索层优化？"

full_chars = len(full_history)
summary_chars = len(summary)
full_tokens = len(enc.encode(full_history))
summary_tokens = len(enc.encode(summary))
saved_tokens = full_tokens - summary_tokens

system_text = f"你是架构顾问。参考对话历史：\n{summary}"
messages = [
    {"role": "system", "content": system_text},
    {"role": "user", "content": next_question},
]

print("=== 完整历史 ===")
for line in RAW_HISTORY:
    print(line)
print(f"{full_chars} 字 / {full_tokens} tokens")

print("\n=== 摘要 buffer ===")
print(summary)
print(f"{summary_chars} 字 / {summary_tokens} tokens")

print("\n=== Token 对比 ===")
print(f"完整历史:  {full_chars} 字 / {full_tokens} tokens")
print(f"摘要 buffer: {summary_chars} 字 / {summary_tokens} tokens")
print(f"节省:      {full_chars - summary_chars} 字 / {saved_tokens} tokens")

print("\n=== Prompt messages ===")
print(f"[system] {system_text}")
print(f"[user] {next_question}")

response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,
)
answer = (response.choices[0].message.content or "").strip()

print("\n=== LLM 回答 ===")
print(answer)

memory.save_context({"input": next_question}, {"output": answer})
RAW_HISTORY.append(f"用户: {next_question}")
RAW_HISTORY.append(f"助手: {answer}")

print("\n=== 第 4 轮对话后 · memory.buffer ===")
print(memory.buffer)
print(f"{len(memory.buffer)} 字 / {len(enc.encode(memory.buffer))} tokens")

print("\n[Part 5] 通过 ✓")

=== 完整历史 ===
用户: 我在做电商客服项目。
助手: 好的，需要我帮你梳理架构吗？
用户: 我们使用向量库存 FAQ。
助手: 明白，可再加一层重排。
用户: 召回后还要做 MMR 去重。
助手: 可以，避免重复段落占满上下文。
用户: 下一步我该优先优化哪一层？
助手: 基于您的电商客服项目现状（向量库检索 + MMR 去重 + 重排），建议按以下优先级优化：

**第一优先级：重排层（Reranker）**
- 原因：MMR 已解决重复问题，但重排能显著提升最终排序质量，直接改善客服回答的准确性
- 具体建议：使用交叉编码器（Cross-encoder）模型，对 MMR 去重后的候选集进行精细排序
- 预期收益：Top-1 准确率可提升 10-20%

**第二优先级：检索层优化**
- 可考虑：混合检索（向量 + 关键词 BM25），弥补纯向量检索对精确匹配的不足
- 或：多向量检索（如 ColBERT），提升细粒度匹配能力

**第三优先级：上下文管理**
- 动态调整 MMR 的多样性权重，根据对话轮次自适应
- 或：引入对话历史压缩，避免长上下文干扰

建议先从重排层入手，这是当前架构中收益最直接、成本可控的优化点。需要我帮您设计重排层的具体实现方案吗？
533 字 / 518 tokens

=== 摘要 buffer ===
用户在做电商客服项目，助手询问是否需要帮忙梳理架构。用户表示使用向量库存 FAQ，助手建议可再加一层重排。用户补充召回后需做 MMR 去重，助手表示同意，认为可避免重复段落占满上下文。随后用户询问优化优先级，助手建议优先优化重排层（使用交叉编码器，预期 Top-1 准确率提升 10-20%），其次优化检索层（混合检索或多向量检索），最后优化上下文管理（动态调整 MMR 权重或对话历史压缩），并询问是否需要设计重排层实现方案。
214 字 / 215 tokens

=== Token 对比 ===
完整历史:  533 字 / 518 tokens
摘要 buffer: 214 字 / 215 tokens
节省:      319 字 / 303 tokens

=== Prompt messages ===
[system] 你是架构顾问。参考对话历史：
用户在做电商客服项目，助手询问是否需要帮忙梳理架构。用户表示

: 